In [ ]:
import os
import bs4
from dotenv import load_dotenv
from langchain_community.embeddings import OpenAIEmbeddings, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.openai_functions.extraction import create_extraction_chain_pydantic
from langchain_classic.chains.sql_database.query import create_sql_query_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
## Data Ingestion and APIs with LangChain

text_loader = TextLoader("../3-rags/sample.txt")
text_documents = text_loader.load()

print(f"All text documents: {text_documents}\n")
print(f"\nLoaded {len(text_documents)} documents.\n")
print(text_documents[0].page_content[:500])  # Print the first 500 characters

All text documents: [Document(metadata={'source': '../3-rags/sample.txt'}, page_content='**Title: The Power and Promise of Machine Learning**\n\nGood morning everyone,\n\nToday, I’d like to talk about something that is quietly transforming our world—machine learning. Machine learning is a branch of artificial intelligence that allows computers to learn from data and improve their performance without being explicitly programmed. \nInstead of telling a computer exactly what to do, we give it data and let it discover patterns on its own.\n\nThink about how you use your phone every day. When your email filters out spam, when Netflix recommends a movie you end up loving, or when your phone unlocks using your face—that’s machine learning at work. It’s not magic; it’s data, algorithms, and continuous learning.\n\nAt its core, machine learning works in three simple steps:\nFirst, data is collected. This could be anything from images and text to numbers and sounds.\nSecond, a model is trained o

In [ ]:
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OLLAMA_API_KEY"] = os.getenv("OLLAMA_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# web based loader
# load, chunk and index content of html page
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
web1_loader = WebBaseLoader(web_path=(url,), bs_kwargs={"parse_only": bs4.SoupStrainer("div", {"class": "mw-content-ltr mw-parser-output"})})
web2_loader = WebBaseLoader(web_path=(url,), bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_="mw-content-ltr mw-parser-output")))

web1_documents = web1_loader.load()
web2_documents = web2_loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
web1_documents

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence'}, page_content='Intelligence of machines\n"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).\n\n\nPart of a series onArtificial intelligence (AI)\nMajor goals\nArtificial general intelligence\nIntelligent agent\nRecursive self-improvement\nPlanning\nComputer vision\nGeneral game playing\nKnowledge representation\nNatural language processing\nRobotics\nAI safety\n\nApproaches\nMachine learning\nSymbolic\nDeep learning\nBayesian networks\nEvolutionary algorithms\nHybrid intelligent systems\nSystems integration\nOpen-source\nAI data centers\n\nApplications\nBioinformatics\nDeepfake\nEarth sciences\n Finance \nGenerative AI\nArt\nAudio\nMusic\nGovernment\nHealthcare\nMental health\nIndustry\nSoftware development\nTranslation\n Military \nPhysics\nProjects\n\nPhilosophy\nAI alignment\nArtificial consciousness\nThe bitter lesson\nChinese room\nFriendly

In [5]:
web1_documents[0].page_content[:500]

'Intelligence of machines\n"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).\n\n\nPart of a series onArtificial intelligence (AI)\nMajor goals\nArtificial general intelligence\nIntelligent agent\nRecursive self-improvement\nPlanning\nComputer vision\nGeneral game playing\nKnowledge representation\nNatural language processing\nRobotics\nAI safety\n\nApproaches\nMachine learning\nSymbolic\nDeep learning\nBayesian networks\nEvolutionary algorithms\nHybrid intelligen'

In [ ]:
pdf_loader = PyPDFLoader(".../3-rags/report.pdf")
pdf_documents = pdf_loader.load()

pdf_mupdf_loader = PyMuPDFLoader("report.pdf")
pdf_mupdf_documents = pdf_mupdf_loader.load()

print(f"Loaded {len(pdf_documents)} documents with PyPDFLoader.")
print(f"Loaded {len(pdf_mupdf_documents)} documents with PyMuPDFLoader.")

Loaded 14 documents with PyPDFLoader.
Loaded 14 documents with PyMuPDFLoader.


In [7]:
pdf_documents[0].page_content[:500]

'Improving  Logistics  Efficiency  by  Enhancing  \nCustomer\n \nRelationship\n \nManagement\n \nUsing\n \nData\n \nAnalytics\n \nin\n \nLPC\n \nGlobal\n  \nPrepared  by:  \nJeremiah  Katumo  Kurwa  (Data  Analyst)  \n \nDate:  \n21\n ST  \nMarch,  2026  \n \nSubmitted  To:  \nHuman  Resource  Officer  -  LPC  Global'

In [8]:
pdf_documents

[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Improving Logistics Efficiency by Enhancing Customer Relationship Management Using Data Analytics in LPC Global', 'source': 'report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='Improving  Logistics  Efficiency  by  Enhancing  \nCustomer\n \nRelationship\n \nManagement\n \nUsing\n \nData\n \nAnalytics\n \nin\n \nLPC\n \nGlobal\n  \nPrepared  by:  \nJeremiah  Katumo  Kurwa  (Data  Analyst)  \n \nDate:  \n21\n ST  \nMarch,  2026  \n \nSubmitted  To:  \nHuman  Resource  Officer  -  LPC  Global'),
 Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Improving Logistics Efficiency by Enhancing Customer Relationship Management Using Data Analytics in LPC Global', 'source': 'report.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2'}, page_content='2  \nTABLE  OF  CONTENT  \nPrepared

In [9]:
pdf_documents[0].page_content

'Improving  Logistics  Efficiency  by  Enhancing  \nCustomer\n \nRelationship\n \nManagement\n \nUsing\n \nData\n \nAnalytics\n \nin\n \nLPC\n \nGlobal\n  \nPrepared  by:  \nJeremiah  Katumo  Kurwa  (Data  Analyst)  \n \nDate:  \n21\n ST  \nMarch,  2026  \n \nSubmitted  To:  \nHuman  Resource  Officer  -  LPC  Global'

In [10]:
# print 3rd page content
pdf_documents[2].page_content[:500]

'3  \n1.  Executive  Summary  \nLPC  Global  is  experiencing  rapid  growth  driven  by  e-commerce  expansion,  urbanization,  and  \nincreased\n \ndemand\n \nfor\n \nefficient\n \nsupply\n \nchain\n \nsolutions.\n \nAs\n \ncompetition\n \nintensifies\n \nfrom\n \nother\n \ncompanies,\n \ncustomer\n \nexperience\n \nhas\n \nbecome\n \na\n \nkey\n \ndifferentiator.\n \nThe\n \ncompany\n \nis\n \nno\n \nlonger\n \nevaluated\n \nsolely\n \non\n \ndelivery\n \nspeed,\n \nbut\n \nalso\n \non\n \ncommunication,\n \nreliability,\n \ntransparency,\n \nand\n \nresponsiveness.\n \n'

In [ ]:
## Transformation

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(pdf_documents)
documents[:5]

[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Improving Logistics Efficiency by Enhancing Customer Relationship Management Using Data Analytics in LPC Global', 'source': 'report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='Improving  Logistics  Efficiency  by  Enhancing  \nCustomer\n \nRelationship\n \nManagement\n \nUsing\n \nData\n \nAnalytics\n \nin\n \nLPC\n \nGlobal\n  \nPrepared  by:  \nJeremiah  Katumo  Kurwa  (Data  Analyst)  \n \nDate:  \n21\n ST  \nMarch,  2026  \n \nSubmitted  To:  \nHuman  Resource  Officer  -  LPC  Global'),
 Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Improving Logistics Efficiency by Enhancing Customer Relationship Management Using Data Analytics in LPC Global', 'source': 'report.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2'}, page_content='2  \nTABLE  OF  CONTENT  \nPrepared

In [ ]:
## Vector Database: Create vector database from documents and perform similarity search

vector_db = Chroma.from_documents(documents[:20], OllamaEmbeddings())

query = "Who is the author of Sapiens novel?"

result = vector_db.similarity_search(query=query)

result

In [ ]:
result[0].page_content

In [ ]:
## FAISS Vector Database

faiss_db = FAISS.from_documents(documents[:20], OllamaEmbeddings())

query = "Who is the author of Sapiens novel?"

result = faiss_db.similarity_search(query=query)

result

In [ ]:
result[0].page_content

In [ ]:
llm = OllamaLLM(model="llama2", temperature=0.7)
llm

OllamaLLM(model='llama2')

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Answer the following questions based on the context provided.
Think step by step before giving a detailed answer.
I will tip you $1000 is the user finds the answer helpfully.

<context>
{context}
</context>

Question: {input}
"""
)

In [ ]:
documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
documents_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following questions based on the context provided.\nThink step by step before giving a detailed answer.\nI will tip you $1000 is the user finds the answer helpfully.\n\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| OllamaLLM(model='llama2')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

### Retrievers

- A retriever is an interface that returns documents relevant to an unstructured query. 
- It is used to retrieve relevant documents from a vector database or a knowledge base. 
- The retriever takes a query as input and returns a list of relevant documents that can be used to answer the query.
- A retriever does not need to store the documents in memory, it can retrive the relevant documents from the vector database or knowledge base on demand.
- Vector stores can be used as the backbones of retrivers, they can be used to store the documents and retrieve the relevant documents based on the query.

In [ ]:
"""
A retriever is an interface that returns documents relevant to an unstructured query. 
It is used to retrieve relevant documents from a vector database or a knowledge base. 
The retriever takes a query as input and returns a list of relevant documents that can be used to answer the query.
A retriever does not need to store the documents in memory, it can retrive the relevant documents from the vector database or knowledge base on demand.
Vector stores can be used as the backbones of retrivers, they can be used to store the documents and retrieve the relevant documents based on the query.
"""
retriever = faiss_db.as_retriever()
retriever

In [ ]:
"""
A retrieval chain is a chain that combines a retriever and a document chain.
It takes a user query as input, uses the retriever to retrieve relevant documents, 
and then uses the document chain (llm chain) to process those documents and generate a final answer.
"""
retrieval_chain = create_retrieval_chain(retriever=retriever, combine_documents_chain=documents_chain)
retrieval_chain

In [ ]:
response = retrieval_chain.invoke({"input": "Who is the author of Sapiens novel?"})
response

In [ ]:
response['answer']